# Module 17 — Tracing and observability

**THE ONE IDEA:** **you cannot retrofit an audit trail** — and the same is true of
observability. If it was not recorded while the run happened, the information does not
exist anywhere.

Module 14 traced *one* run for compliance. This widens the same idea into the six things
you actually need per run, then rolls them up:

**intent · tool calls and args · latency · tokens · cost · approvals**

Runs on `_fake_model`, so the trace is deterministic and free.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json, time, pathlib
from contextlib import contextmanager
from _fake_model import FakeModel, tool_turn, text_turn
from _tools import run_tool, WRITE_TOOLS

TRACE = pathlib.Path("traces.jsonl"); TRACE.unlink(missing_ok=True)
PRICE_IN, PRICE_OUT = 0.40, 1.60          # $ per 1M, gpt-4.1-mini

class Tracer:
    """One object per run. Writes append-only, and rolls up at the end."""
    def __init__(self, run_id, intent):
        self.run_id, self.intent, self.spans = run_id, intent, []
        self.t0 = time.time()

    @contextmanager
    def span(self, kind, **meta):
        t = time.time(); rec = {"kind": kind, **meta}
        try: yield rec
        finally:
            rec["ms"] = round((time.time() - t) * 1000, 1)
            self.spans.append(rec)

    def close(self, outcome):
        n = lambda k: sum(s["kind"] == k for s in self.spans)
        tin  = sum(s.get("tokens_in", 0)  for s in self.spans)
        tout = sum(s.get("tokens_out", 0) for s in self.spans)
        s = {"run_id": self.run_id, "intent": self.intent, "outcome": outcome,
             "wall_ms": round((time.time() - self.t0) * 1000, 1),
             "llm_turns": n("llm"), "tool_calls": n("tool"), "approvals": n("approval"),
             "tokens_in": tin, "tokens_out": tout,
             "cost_usd": round(tin * PRICE_IN / 1e6 + tout * PRICE_OUT / 1e6, 6),
             "spans": self.spans}
        with TRACE.open("a") as f: f.write(json.dumps(s) + "\n")
        return s

## The traced loop

In [ ]:
def traced_run(run_id, intent, script, approve=lambda n, a: True):
    tr, fake = Tracer(run_id, intent), FakeModel(script)
    messages = [{"role": "user", "content": intent}]
    for step in range(1, 7):
        with tr.span("llm", step=step) as s:
            r = fake.create(messages=messages)
            s["tokens_in"]  = r.usage.prompt_tokens
            s["tokens_out"] = r.usage.completion_tokens
            s["finish"] = r.choices[0].finish_reason
        msg = r.choices[0].message
        if r.choices[0].finish_reason != "tool_calls":
            return tr.close("answered")
        for tc in msg.tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments)
            if name in WRITE_TOOLS:
                with tr.span("approval", tool=name, args=args) as s:
                    s["approved"] = approve(name, args); s["approver"] = "s.khan"
                if not s["approved"]:
                    with tr.span("tool_blocked", tool=name): pass
                    messages.append({"role": "user", "content": "DENIED by a human."})
                    continue
            with tr.span("tool", tool=name, args=args, write=name in WRITE_TOOLS) as s:
                s["result"] = run_tool(name, args)[:80]
            messages.append({"role": "user", "content": s["result"]})
    return tr.close("capped")

## Three runs with different shapes

In [ ]:
SHORT = [tool_turn("search_policy", {"query": "erc"}, "a"), text_turn("ERC year 2 is 4%.")]
LONG  = [tool_turn("search_policy", {"query": "erc"}, "a"),
         tool_turn("calculate", {"expression": "250000*0.04"}, "b"),
         tool_turn("search_policy", {"query": "ltv"}, "c"),
         tool_turn("search_policy", {"query": "rates"}, "d"), text_turn("10000.")]
WRITE = [tool_turn("fetch_customer_note", {"customer_id": "C-1002"}, "a"),
         tool_turn("confirm_decision", {"reference": "W-1"}, "b"), text_turn("Recorded.")]

runs = [traced_run("r-short", "What is the year-2 ERC?", SHORT),
        traced_run("r-wander", "Year-2 ERC on 250000?", LONG),
        traced_run("r-write", "Approve waiver for C-1002.", WRITE, approve=lambda n, a: False)]
for r in runs: print(f"{r['run_id']:9} {r['outcome']:9} {r['llm_turns']} turns")

## The roll-up — what a dashboard actually shows

In [ ]:
print(f"{'run':9} {'turns':>5} {'tools':>5} {'appr':>4} {'in':>6} {'out':>5} "
      f"{'ms':>7} {'cost $':>9}")
print("-" * 60)
for r in runs:
    print(f"{r['run_id']:9} {r['llm_turns']:5} {r['tool_calls']:5} {r['approvals']:4} "
          f"{r['tokens_in']:6} {r['tokens_out']:5} {r['wall_ms']:7.1f} {r['cost_usd']:9.6f}")
print("-" * 60)

slow = max(runs, key=lambda r: r["cost_usd"])
print(f"\nmost expensive run: {slow['run_id']} at ${slow['cost_usd']:.6f} — "
      f"{slow['tool_calls']} tool calls")
print("drill into it:", [s.get("tool") for s in slow["spans"] if s["kind"] == "tool"])

print("""
LESSON - six fields, captured DURING the run, and you can answer every question
an on-call engineer or an auditor will ask:

  intent      what was this run FOR?            -> triage without guessing
  tool calls  which tools, with what args?      -> the FM3 wander is visible above
  latency     per span, not just end to end     -> find the slow tool, not the slow run
  tokens      in and out, per turn              -> module 11's growth curve, live
  cost        per run, roll up by day or user   -> the number that gets you shut down
  approvals   who said yes, to what, when       -> module 14's compliance requirement

None of these can be reconstructed afterwards from a final answer. The r-wander
run cost 4x r-short for the same question, and the ONLY reason you can see that
is that someone recorded it at the time.

In production you would not hand-roll this - LangFuse, LangSmith or Phoenix all
speak OpenTelemetry and give you the dashboard. Build it once by hand first so
you know what those tools are actually storing, and what they cannot recover.""")

---

**Next:** Block G — `../G_frameworks/18_langchain_lcel.ipynb`